# Task 03 — Gold: Daily Revenue Aggregation

**Workshop: Final Pipeline | Layer 3 of 3**

## Goal

Read `silver.lab_orders`, aggregate to daily metrics, write to `gold.lab_daily_orders`.

```
silver.lab_orders
        |
        v  to_date * groupBy * agg
        v
gold.lab_daily_orders
```

## Expected output schema

| Column | Type | Description |
|--------|------|-------------|
| order_date | DateType | Calendar day from `order_datetime` |
| order_count | LongType | Total orders that day |
| total_revenue | DoubleType | SUM of `net_amount` |
| avg_order_value | DoubleType | AVG of `net_amount`, rounded to 2 dp |

> Use `net_amount` (after discount), not `total_amount`. One row = one calendar day.


In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")
dbutils.widgets.text("gold_schema",   GOLD_SCHEMA,   "Gold Schema")

catalog       = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema   = dbutils.widgets.get("gold_schema")

source_table = f"{catalog}.{silver_schema}.lab_orders"
target_table = f"{catalog}.{gold_schema}.lab_daily_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import aggregate functions and `to_date` from `pyspark.sql.functions`.

> Note: Python has built-in `sum()`, `round()`, and `min()`.
> Importing the same names from `pyspark.sql.functions` shadows them here — that is intentional.


In [ ]:
# HINT -- aggregate functions:
#
#   from pyspark.sql.functions import col, to_date, count, sum, avg, round
#
#   count("*")         -> all rows in the group (includes nulls in other cols)
#   count("col_name")  -> non-null values in that column
#   sum("col_name")    -> sum of non-null values
#   avg("col_name")    -> mean of non-null values
#
#   round(column_expression, scale)
#       -> rounds a Column expression to N decimal places
#          The first argument must be a Column, not a Python float.
#          Correct:   round(avg("net_amount"), 2)
#          Incorrect: round(2.345, 2)   <- Python built-in, not Spark
#
#   to_date(timestamp_column)
#       -> extracts DateType from TimestampType
#          2024-03-15T14:23:00 -> 2024-03-15 (time removed)

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Read the Silver table


In [ ]:
# HINT -- spark.table():
#
#   df = spark.table("catalog.schema.table_name")
#
#   Task 02 must have run first — this table must already exist.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.printSchema()

---

## Exercise 3 — Extract `order_date` from `order_datetime`

Add a new column `order_date` (DateType) by extracting the calendar date
from `order_datetime` (TimestampType).


In [ ]:
# HINT -- to_date():
#
#   to_date(col("timestamp_column"))
#       -> DateType: year-month-day; time part removed
#
#   Add it as a new column:
#       df.withColumn("order_date", to_date(col("order_datetime")))
#
#   You can do this here or inline it as the first step in groupBy.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 4 — Aggregate: one row per day

Group by `order_date`, compute the four metrics, sort ascending by date.
Name the result `daily_df`.


In [ ]:
# HINT -- groupBy + agg():
#
#   daily_df = (
#       df
#       .groupBy("grouping_column")
#       .agg(
#           count("*").alias("alias"),
#           sum("numeric_col").alias("alias"),
#           round(avg("numeric_col"), 2).alias("alias"),
#       )
#       .orderBy("grouping_column")
#   )
#
#   .agg() accepts any number of Column expressions.
#   Always use .alias("name") — without it the column is named
#   "sum(net_amount)" etc., which is awkward downstream.
#
#   round(avg(...), 2): avg() returns a Column; round() wraps it.
#   The nesting is intentional.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Daily rows : {daily_df.count():,}  (= distinct order dates)")
daily_df.show(15, truncate=False)

---

## Exercise 5 — Write to Gold as a managed Delta table


In [ ]:
# HINT -- write to Gold:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")       # Gold is always full-refresh
#       .saveAsTable("catalog.schema.table_name")
#
#   Gold is always rebuilt from Silver on every run.
#   No .option("overwriteSchema") needed — Gold schema is stable.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} daily records")
display(spark.table(target_table))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))